# *Nonlinear Arterial Hemodynamics*
## Chapter 8 companion — Computing the Theory

This notebook is the computational and verification companion to Chapter 8.

Its purpose is not to introduce a new hemodynamic mechanism. It reconstructs the numerical machinery required by the preceding theory and makes the verification contract executable:

- harmonic reconstruction and complex Bessel evaluation;
- coupled radial boundary-value solution;
- nonlinear real-field reconstruction;
- Fourier pseudospectral evolution with de-aliasing;
- second-order streaming solution;
- geometry-correct quadrature;
- convergence, limiting-case recovery, mechanism-off tests, and provenance.

The book nomenclature governs all reader-facing quantities.

The notebook also records one source-level numerical inconsistency found during implementation: the Chapter 8 prose specifies a fourth-order finite-difference streaming solve, while the published Case C refinement sequence is reproduced by the lower-order centered radial discretization used as an independent cross-check in the Chapter 6 notebook. Both paths are retained here and clearly distinguished.

**Execution:** a clean Google Colab runtime should reproduce all outputs using **Run all** with no manual parameter choices.

# Chapter question

Chapter 8 asks:

> What numerical machinery is required to reproduce and test the preceding theory without obscuring its mechanics?

The answer is a verification hierarchy rather than a software framework.

Every workflow begins with three representation choices:

1. **What field is represented?**
2. **In which coordinate is that field smooth or periodic?**
3. **Which exact or limiting solution must be recovered?**

The notebook implements the three canonical workflows:

- **Case A:** constitutive coupling — adaptive radial BVP with isotropic recovery;
- **Case B:** spectral redistribution — split Fourier pseudospectral evolution with $2/3$ de-aliasing and energy diagnostics;
- **Case C:** compliant streaming — analytical first-order mode plus radial second-order mean solve, moving-wall closure, flux, and traction.

The VascuQuest section is separate. It demonstrates deterministic source-to-book mapping and provenance but does not replace the canonical verification cases.

# Book mechanics used here

## Representation contract

The numerical object must remain attached to the physical quantity defined by the derivation.

Examples are:

$$
\widehat u_m(r),
$$

$$
(\widehat u_{z,m},\widehat u_{\theta,m}),
$$

$$
\widetilde a(\zeta,s),
$$

and

$$
\langle u_{2z}(r)\rangle.
$$

Chebyshev or adaptive boundary-value methods are natural for smooth radial problems on a finite interval. Fourier methods are natural for periodic axial evolution. Uniform radial finite differences are used for the streaming boundary-value problem.

## Verification versus validation

The canonical checks in this notebook are primarily **verification and internal-consistency tests**. They establish that the intended mathematical problems are being solved correctly.

External validation would require independently measured physical data and uncertainty accounting for the same operationally defined variables. VascuQuest is therefore used here as a reproducible source of virtual-population inputs and mapping examples, not as external experimental validation.

# VascuQuest representation

VascuQuest/PWDB is used only to demonstrate reproducible data access and the translation from source-native quantities to book quantities.

For a deterministic representative subject and selected arterial sites, the notebook records:

- source age;
- heart rate;
- luminal-area waveform;
- flow-velocity waveform;
- pressure waveform;
- source geometry.

The mapping layer constructs

$$
Q(t)=U(t)A(t),
$$

$$
R=\sqrt{\frac{\langle A\rangle_t}{\pi}},
$$

$$
\Omega=\frac{2\pi}{T},
$$

$$
\alpha=R\sqrt{\frac{\Omega}{\nu}}.
$$

Database-native field names are confined to the ingestion code.

This section is a reproducibility demonstration. It does not turn the Case A–C verification settings into physiological ranges.

In [ ]:
# Configuration and canonical benchmark constants
from pathlib import Path
import sys, json, subprocess, zipfile, math

ROOT = Path("/content/nonlinear_arterial_hemodynamics_ch08")
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
META_DIR = ROOT / "metadata"
for d in (ROOT, FIG_DIR, DATA_DIR, META_DIR):
    d.mkdir(parents=True, exist_ok=True)

VQ_REPOSITORY = "https://github.com/KNOWDYN/VascuQuest.git"
VQ_GIT_REF = "8307147d72e7a6f3ea3135895bd6f52927c67439"
PWDB_RECORD_ID = "3275625"
PWDB_DOI = "10.5281/zenodo.3275625"

rho = 1060.0
mu = 3.5e-3
nu = mu/rho

# Case A
alpha_A = 8.0
m_A = 1
a_A = 1.0
A_ztheta = 0.1
A_thetaz = 0.1
A_thetatheta = 1.0

# Case B
L_g = 4.0*3.141592653589793
alpha_B = 10.0
b_B = alpha_B**-2
g_B = 0.005*(1.0 + 0.1/alpha_B)
kappa_c = 2.0

# Case C
R_C = 4e-3
f_C = 1.2
Omega_C = 2.0*3.141592653589793*f_C
kR_C = 0.2
k_C = kR_C/R_C
P_C = 1.0

print("Working directory:", ROOT)

In [ ]:
# Install the pinned VascuQuest revision and import numerical libraries.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    f"git+{VQ_REPOSITORY}@{VQ_GIT_REF}"
])

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.integrate import solve_bvp
from scipy.special import jv, iv
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve
import vascuquest as vq

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("VascuQuest:", getattr(vq, "__version__", "version field not exposed"))

In [ ]:
# Acquire and checksum-verify the PWDB artifacts used by the reproducibility example.
ARTIFACTS = [
    "model_configurations",
    "common_site_waveforms_csv",
    "geometry",
]
verification = {}

for artifact in ARTIFACTS:
    subprocess.run(
        ["vascuquest", "dataset", "acquire",
         "--artifact", artifact, "--yes", "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verified = subprocess.run(
        ["vascuquest", "dataset", "verify",
         "--artifact", artifact, "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verification[artifact] = json.loads(verified.stdout)

status = subprocess.run(
    ["vascuquest", "dataset", "status", "--format", "json"],
    check=True, text=True, capture_output=True,
)
dataset_status = json.loads(status.stdout)
SOURCE_DIR = Path(dataset_status["managed_paths"]["source"])

(META_DIR / "artifact_verification.json").write_text(
    json.dumps(verification, indent=2), encoding="utf-8"
)
(META_DIR / "dataset_status.json").write_text(
    json.dumps(dataset_status, indent=2), encoding="utf-8"
)

print("Verified PWDB source:", SOURCE_DIR)

In [ ]:
# Deterministic subject metadata and representative-subject rule.
session = vq.open_dataset(source=SOURCE_DIR, offline=True)
assert session.identity.record_id == PWDB_RECORD_ID

age_result = session.get("age")
subject_ids = np.asarray(age_result.coordinates[0].values, dtype=str)
ages = np.asarray(age_result.values, dtype=float)

hr_result = session.get("heart_rate", subjects=subject_ids.tolist())
hr_ids = np.asarray(hr_result.coordinates[0].values, dtype=str)
heart_rates = np.asarray(hr_result.values, dtype=float)
assert np.array_equal(subject_ids, hr_ids)

subject_meta = pd.DataFrame({
    "subject_id": subject_ids,
    "age_years": ages,
    "heart_rate_bpm": heart_rates,
})
subject_meta["subject_number"] = subject_meta["subject_id"].astype(int)
subject_meta = subject_meta.sort_values("subject_number").reset_index(drop=True)

source_ages = sorted(subject_meta["age_years"].dropna().unique())
target_age = source_ages[len(source_ages)//2]
group = subject_meta.loc[subject_meta["age_years"] == target_age].copy()
group = group.sort_values("subject_number").reset_index(drop=True)
representative_subject = str(group.iloc[len(group)//2]["subject_id"])

selection_record = {
    "rule": "middle PWDB source age stratum; median canonical subject number",
    "representative_subject_id": representative_subject,
    "representative_age_years": float(target_age),
}
(META_DIR / "representative_subject.json").write_text(
    json.dumps(selection_record, indent=2), encoding="utf-8"
)
display(pd.DataFrame([selection_record]))

In [ ]:
# Shared plotting and source-mapping utilities.
WAVE_ZIP = SOURCE_DIR / "PWs_csv.zip"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9.0,
    "axes.labelsize": 9.0,
    "axes.titlesize": 9.5,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 7.8,
    "axes.linewidth": 0.75,
    "lines.linewidth": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
BLACK, DARK, MID, LIGHT = "0.0", "0.28", "0.52", "0.74"

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

def save_figure(fig, stem):
    pdf = FIG_DIR / f"{stem}.pdf"
    png = FIG_DIR / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(png, dpi=600, bbox_inches="tight", pad_inches=0.03)
    return pdf, png

def _wave_member_name(site_id, source_signal):
    basename = f"PWs_{site_id}_{source_signal}.csv"
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        matches = [name for name in zf.namelist() if Path(name).name == basename]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {basename!r}; found {len(matches)}")
    return matches[0]

def load_waveform_matrix(site_id, source_signal):
    member = _wave_member_name(site_id, source_signal)
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        with zf.open(member, "r") as raw:
            frame = pd.read_csv(raw, low_memory=False)
    ids = np.asarray([str(int(x)) for x in frame.iloc[:, 0].to_numpy()], dtype=str)
    values = frame.iloc[:, 1:].to_numpy(dtype=float)
    return ids, values

def contiguous_prefix(*rows):
    finite = np.ones_like(np.asarray(rows[0], dtype=float), dtype=bool)
    for row in rows:
        finite &= np.isfinite(np.asarray(row, dtype=float))
    bad = np.flatnonzero(~finite)
    stop = int(bad[0]) if len(bad) else len(finite)
    if np.any(finite[stop:]):
        raise ValueError("Internal missing samples would alter waveform phase.")
    if stop < 16:
        raise ValueError("Insufficient contiguous waveform samples.")
    return tuple(np.asarray(row, dtype=float)[:stop] for row in rows)

print("Shared utilities ready.")

# Fourier reconstruction of prescribed waveforms

The book convention is

$$
G(t)
=
G_0
+
\Re\left\{
\sum_{m=1}^{M}
\widehat G_m e^{im\Omega t}
\right\}.
$$

A discrete transform must use the same normalization when reconstructing velocity, vorticity, stress, and nonlinear products.

The truncation level $M$ controls the prescribed waveform. A quadratic observable may contain frequencies up to $2M$, so waveform convergence and nonlinear-observable convergence are separate checks.

In [ ]:
# Deterministic transform-normalization and reconstruction test.
phase = np.linspace(0.0, 1.0, 1024, endpoint=False)
signal = (
    2.0
    + 1.2*np.cos(2*np.pi*phase+0.3)
    + 0.4*np.cos(4*np.pi*phase-0.7)
    + 0.15*np.cos(6*np.pi*phase+0.2)
)

c = np.fft.rfft(signal)/len(signal)
mean = c[0].real
M = 3
hat = 2.0*c[1:M+1]
m = np.arange(1, M+1)

reconstructed = mean + np.real(
    np.exp(2j*np.pi*np.outer(phase, m)) @ hat
)

reconstruction_error = np.max(np.abs(reconstructed-signal))
assert reconstruction_error < 1e-12

quadratic = reconstructed**2
cq = np.fft.rfft(quadratic)/len(quadratic)
ampq = 2.0*np.abs(cq)
ampq[0] = np.abs(cq[0])

fig, axes = plt.subplots(1, 2, figsize=(7.1, 3.05))

axes[0].plot(phase, signal, color=LIGHT, linestyle=":", label="source")
axes[0].plot(phase, reconstructed, color=BLACK, label="reconstruction")
axes[0].set_xlabel(r"Normalized time, $t/T$")
axes[0].set_ylabel("Signal")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].stem(
    np.arange(min(8, len(ampq))),
    ampq[:8],
    linefmt="k-", markerfmt="ko", basefmt=" "
)
axes[1].axvline(M, color=LIGHT, linestyle=":", linewidth=1.0)
axes[1].set_xlabel(r"Output harmonic, $m$")
axes[1].set_ylabel("Quadratic-spectrum amplitude")
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch08_fourier_reconstruction_contract")
plt.show()

print("Maximum reconstruction error:", reconstruction_error)

The left panel verifies the transform convention itself. The right panel shows that the nonlinear observable contains output frequencies beyond the highest retained input harmonic.

This is why the completed real fields must be reconstructed before nonlinear multiplication, or equivalently why the full convolution must be evaluated.

# Complex Bessel evaluation of the classical radial mode

For harmonic $m$,

$$
\widehat u_m(r)
=
\frac{\widehat G_m}{im\Omega\rho}
\left[
1-
\frac{J_0(\Lambda_{W,m}r/R)}
{J_0(\Lambda_{W,m})}
\right],
$$

with

$$
\Lambda_{W,m}=i^{3/2}\sqrt m\,\alpha.
$$

The analytical radial derivative is

$$
\frac{d\widehat u_m}{dr}
=
\frac{\widehat G_m}{im\Omega\rho}
\frac{\Lambda_{W,m}}{R}
\frac{J_1(\Lambda_{W,m}r/R)}
{J_0(\Lambda_{W,m})}.
$$

Analytical differentiation provides a reference against which numerical differentiation can be checked.

In [ ]:
# Exact Womersley function and derivative check.
def Lambda_W(alpha_m):
    return np.exp(3j*np.pi/4.0)*alpha_m

def womersley_hat(r, R, Omega, alpha, Ghat=1.0, m=1):
    LW = Lambda_W(np.sqrt(m)*alpha)
    return (
        Ghat/(1j*m*Omega*rho)
        * (1.0-jv(0, LW*r/R)/jv(0, LW))
    )

def womersley_derivative(r, R, Omega, alpha, Ghat=1.0, m=1):
    LW = Lambda_W(np.sqrt(m)*alpha)
    return (
        Ghat/(1j*m*Omega*rho)
        * (LW/R)
        * jv(1, LW*r/R)/jv(0, LW)
    )

R_ref = 4e-3
Omega_ref = 2*np.pi*1.2
alpha_ref = R_ref*np.sqrt(Omega_ref/nu)

r = np.linspace(0.0, R_ref, 1400)
u = womersley_hat(r, R_ref, Omega_ref, alpha_ref)
du_exact = womersley_derivative(r, R_ref, Omega_ref, alpha_ref)
du_numeric = np.gradient(u, r, edge_order=2)

mask = (r > 0.02*R_ref) & (r < 0.98*R_ref)
derivative_error = (
    np.max(np.abs(du_numeric[mask]-du_exact[mask]))
    / np.max(np.abs(du_exact[mask]))
)

fig, ax = plt.subplots(figsize=(5.9, 3.25))
ax.plot(r/R_ref, np.abs(du_exact), color=BLACK, label="analytical derivative")
ax.plot(r/R_ref, np.abs(du_numeric), color=DARK, linestyle="--",
        label="numerical derivative")
ax.set_xlabel(r"Normalized radius, $r/R$")
ax.set_ylabel(r"$|d\widehat u/dr|$")
ax.legend(frameon=False)
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch08_bessel_derivative_check")
plt.show()

print("Interior relative derivative error:", derivative_error)

# Case A — constitutive coupling

The canonical dimensionless parameters are

$$
\alpha=8,
\qquad
m=1,
\qquad
a_m=1,
$$

$$
\mathcal A_{z\theta}
=
\mathcal A_{\theta z}
=
0.1,
\qquad
\mathcal A_{\theta\theta}=1.
$$

Chapter 8 verifies the differential system with an adaptive boundary-value solver and reports the tolerance sequence

$$
10^{-5},\;10^{-7},\;10^{-9}.
$$

The strongest physical counterfactual is the isotropic limit:

$$
\mathcal A_{z\theta}
=
\mathcal A_{\theta z}
=
0
\quad\Longrightarrow\quad
U_\theta\rightarrow0
$$

and the axial field returns to the scalar Womersley solution.

In [ ]:
# Adaptive radial BVP used for Case A.
def solve_caseA(
    tol,
    Azt=A_ztheta,
    Atz=A_thetaz,
    Att=A_thetatheta,
    epsilon=1e-5,
):
    coupling = np.array([[1.0, Azt], [Atz, Att]], dtype=float)
    inv_coupling = np.linalg.inv(coupling)
    x = np.linspace(epsilon, 1.0, 300)
    lam = m_A*alpha_A**2

    def ode(x, y):
        Uz = y[0] + 1j*y[4]
        Uzp = y[1] + 1j*y[5]
        Uth = y[2] + 1j*y[6]
        Uthp = y[3] + 1j*y[7]

        rhs_z = 1j*lam*Uz - a_A - Uzp/x
        rhs_th = (
            1j*lam*Uth
            - 2.0*Atz*Uzp/x
            - Att*Uthp/x
            + Att*Uth/x**2
        )

        second = inv_coupling @ np.vstack([rhs_z, rhs_th])
        Uzpp, Uthpp = second[0], second[1]

        out = np.empty_like(y)
        vals = [Uzp, Uzpp, Uthp, Uthpp]
        for j, value in enumerate(vals):
            out[j] = value.real
            out[j+4] = value.imag
        return out

    def bc(ya, yb):
        residual = [
            ya[1]+1j*ya[5],
            ya[2]+1j*ya[6],
            yb[0]+1j*yb[4],
            yb[2]+1j*yb[6],
        ]
        return np.array(
            [z.real for z in residual]
            + [z.imag for z in residual]
        )

    y0 = np.zeros((8, len(x)))
    y0[0] = (1.0-x**2)/(1.0+alpha_A**2)
    y0[1] = -2.0*x/(1.0+alpha_A**2)

    sol = solve_bvp(
        ode, bc, x, y0,
        tol=tol, max_nodes=30000
    )
    if sol.status != 0:
        raise RuntimeError(sol.message)
    return sol

def eval_caseA(sol, x):
    y = sol.sol(x)
    return (
        y[0]+1j*y[4],
        y[1]+1j*y[5],
        y[2]+1j*y[6],
        y[3]+1j*y[7],
    )

caseA_rows = []
caseA_solutions = {}

for tol in [1e-5, 1e-7, 1e-9]:
    sol = solve_caseA(tol)
    xx = np.linspace(1e-5, 1.0, 1600)
    Uz, Uzp, Uth, Uthp = eval_caseA(sol, xx)

    caseA_solutions[tol] = sol
    caseA_rows.append({
        "tolerance": tol,
        "adaptive_nodes": len(sol.x),
        "max_abs_Uz": np.max(np.abs(Uz)),
        "max_abs_Utheta": np.max(np.abs(Uth)),
    })

caseA_df = pd.DataFrame(caseA_rows)
caseA_df.to_csv(DATA_DIR / "ch08_caseA_convergence.csv", index=False)
display(caseA_df)

# Book benchmark values.
ref_Uz = 0.017052742
ref_Uth = 4.973726e-4
assert abs(caseA_df.iloc[-1]["max_abs_Uz"]-ref_Uz)/ref_Uz < 2e-5
assert abs(caseA_df.iloc[-1]["max_abs_Utheta"]-ref_Uth)/ref_Uth < 2e-5

In [ ]:
# Case A isotropic counterfactual and field comparison.
sol_an = caseA_solutions[1e-9]
sol_iso = solve_caseA(
    1e-9, Azt=0.0, Atz=0.0, Att=1.0
)

xA = np.linspace(1e-5, 1.0, 1000)
Uz_an, _, Uth_an, _ = eval_caseA(sol_an, xA)
Uz_iso, _, Uth_iso, _ = eval_caseA(sol_iso, xA)

LW = Lambda_W(alpha_A)
Uz_exact_iso = (
    1.0/(1j*alpha_A**2)
    * (1.0-jv(0, LW*xA)/jv(0, LW))
)

axial_recovery = (
    np.max(np.abs(Uz_iso-Uz_exact_iso))
    / np.max(np.abs(Uz_exact_iso))
)
transverse_off = np.max(np.abs(Uth_iso))

fig, axes = plt.subplots(1, 2, figsize=(7.15, 3.05))

axes[0].plot(xA, np.abs(Uz_an), color=BLACK, label="Case A")
axes[0].plot(xA, np.abs(Uz_iso), color=DARK, linestyle="--",
             label="anisotropy off")
axes[0].set_xlabel(r"Normalized radius, $x=r/R$")
axes[0].set_ylabel(r"$|\widehat U_z|$")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].plot(xA, np.abs(Uth_an), color=BLACK, label="Case A")
axes[1].plot(xA, np.maximum(np.abs(Uth_iso),1e-175),
             color=DARK, linestyle="--", label="anisotropy off")
axes[1].set_yscale("log")
axes[1].set_xlabel(r"Normalized radius, $x=r/R$")
axes[1].set_ylabel(r"$|\widehat U_\theta|$")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch08_caseA_verification")
plt.show()

print("Isotropic axial recovery error:", axial_recovery)
print("Isotropic max|U_theta|:", transverse_off)

The convergence sequence verifies numerical consistency. The isotropic counterfactual verifies that the implemented boundary-value problem is the intended constitutive problem.

These are distinct tests and both are required.

# Vorticity and nonlinear reconstruction

For the Chapter 4 kinematics,

$$
\widehat\omega_{\theta,m}
=
-\frac{\partial\widehat u_{z,m}}{\partial r},
$$

$$
\widehat\omega_{z,m}
=
\frac1r
\frac{\partial}{\partial r}
(r\widehat u_{\theta,m}).
$$

The real fields are reconstructed first. Only then is

$$
\boldsymbol\ell
=
\mathbf u\times\boldsymbol\omega
$$

formed.

A subsequent FFT is a diagnostic transformation of the already constructed nonlinear signal.

In [ ]:
# Case A reconstruction-before-multiplication check.
xx = np.linspace(1e-4, 1.0, 700)
Uz, Uzp, Uth, Uthp = eval_caseA(sol_an, xx)

phaseA = np.linspace(0.0, 1.0, 512, endpoint=False)
E = np.exp(2j*np.pi*phaseA)[:,None]

uz = np.real(E*Uz[None,:])
uzp = np.real(E*Uzp[None,:])
uth = np.real(E*Uth[None,:])
uthp = np.real(E*Uthp[None,:])

omega_z = uthp + uth/xx[None,:]
ell_r = uth*omega_z + uz*uzp

j = int(np.argmin(np.abs(xx-0.90)))
sig = ell_r[:,j]
c = np.fft.rfft(sig)/len(sig)
amp = 2.0*np.abs(c)
amp[0] = np.abs(c[0])

fig, ax = plt.subplots(figsize=(5.8,3.2))
ax.stem(
    np.arange(6), amp[:6],
    linefmt="k-", markerfmt="ko", basefmt=" "
)
ax.set_xlabel(r"Output harmonic, $m$")
ax.set_ylabel(r"Spectrum of $\ell_r$ at $x=0.9$")
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch08_nonlinear_reconstruction")
plt.show()

# Case B — Fourier pseudospectral evolution

For the periodic reduced model,

$$
\mathcal L(\kappa)
=
+i b_{\mathrm{avg}}\kappa^3
-
g_{\mathrm{avg}}|\kappa|.
$$

The quadratic term is evaluated in physical space and transformed back to Fourier space. A $2/3$ de-aliasing mask is applied to the nonlinear product.

The canonical parameters are

$$
L_g=4\pi,
\qquad
\alpha=10,
$$

$$
b_{\mathrm{avg}}=0.01,
\qquad
g_{\mathrm{avg}}=0.00505,
$$

with

$$
\widetilde a(\zeta,0)
=
\sin(0.5\zeta)
+
0.3\sin\zeta
+
0.1\sin(1.5\zeta).
$$

In [ ]:
# Split Fourier pseudospectral solver for Case B.
def geometry_grid(N):
    zeta = np.arange(N)*L_g/N
    kappa = 2.0*np.pi*np.fft.fftfreq(N, d=L_g/N)
    mode_number = np.fft.fftfreq(N)*N
    return zeta, kappa, mode_number

def caseB_initial(zeta):
    return (
        np.sin(0.5*zeta)
        + 0.3*np.sin(zeta)
        + 0.1*np.sin(1.5*zeta)
    )

def caseB_diag(a_hat, kappa):
    N = len(a_hat)
    coeff = a_hat/N
    I2 = L_g*np.sum(np.abs(coeff)**2)
    low = (
        (np.abs(kappa)>1e-14)
        & (np.abs(kappa)<=kappa_c+1e-14)
    )
    high = np.abs(kappa)>kappa_c+1e-14
    Elow = L_g*np.sum(np.abs(coeff[low])**2)
    Ehigh = L_g*np.sum(np.abs(coeff[high])**2)
    Dsum = L_g*np.sum(np.abs(kappa)*np.abs(coeff)**2)
    return I2, Elow, Ehigh, Ehigh/Elow, Dsum

def run_caseB(
    N,
    dt,
    s_end=10.0,
    dealias=True,
    nonlinear=True,
    damping=True,
    save_every=None,
):
    zeta, kappa, mode_number = geometry_grid(N)
    a_hat = np.fft.fft(caseB_initial(zeta))

    g_use = g_B if damping else 0.0
    linear = 1j*b_B*kappa**3 - g_use*np.abs(kappa)
    Ehalf = np.exp(linear*dt/2.0)

    mask = np.ones(N, dtype=bool)
    if dealias:
        mask = np.abs(mode_number)<=N/3.0
        a_hat[~mask] = 0.0

    def rhs(h):
        if not nonlinear:
            return np.zeros_like(h)
        a = np.fft.ifft(h).real
        out = -0.5j*kappa*np.fft.fft(a*a)
        if dealias:
            out[~mask] = 0.0
        return out

    nsteps = int(round(s_end/dt))
    if save_every is None:
        save_every = max(1, nsteps//200)

    history = []

    for step in range(nsteps+1):
        if step % save_every == 0 or step == nsteps:
            I2, Elow, Ehigh, Rspec, Dsum = caseB_diag(a_hat, kappa)
            Gamma = -2.0*g_use*Dsum/I2 if I2>0 and damping else 0.0
            history.append((step*dt, I2, Rspec, Gamma))

        if step == nsteps:
            break

        a_hat = Ehalf*a_hat
        if dealias:
            a_hat[~mask] = 0.0

        k1 = rhs(a_hat)
        k2 = rhs(a_hat+0.5*dt*k1)
        k3 = rhs(a_hat+0.5*dt*k2)
        k4 = rhs(a_hat+dt*k3)
        a_hat += dt*(k1+2*k2+2*k3+k4)/6.0

        if dealias:
            a_hat[~mask] = 0.0

        a_hat = Ehalf*a_hat

    return zeta, kappa, a_hat, np.asarray(history)

print("Case B solver ready.")

In [ ]:
# Reproduce the Chapter 8 Case B refinement table.
caseB_rows = []
caseB_results = {}

for N, dt in [(128,4e-3),(256,2e-3),(512,1e-3)]:
    print("Running", N, dt)
    z, kappa, ahat, hist = run_caseB(N, dt)
    caseB_results[N] = (z,kappa,ahat,hist)
    caseB_rows.append({
        "N_zeta": N,
        "Delta_s": dt,
        "I2_10": hist[-1,1],
        "R_spec_10": hist[-1,2],
    })

caseB_df = pd.DataFrame(caseB_rows)
caseB_df.to_csv(DATA_DIR / "ch08_caseB_refinement.csv", index=False)
display(caseB_df)

# Published finest-grid references.
assert abs(caseB_df.iloc[-1]["I2_10"]-4.984669) < 4e-5
assert abs(caseB_df.iloc[-1]["R_spec_10"]-10.11474) < 2e-3

In [ ]:
# Case B de-aliasing and energy-identity checks.
_, _, _, hist256 = caseB_results[256]

_, _, _, hist256_noalias = run_caseB(
    256, 2e-3, dealias=False
)

_, _, _, hist256_nodamp = run_caseB(
    256, 2e-3, damping=False
)

history = pd.DataFrame(
    hist256,
    columns=["s","I2","R_spec","Gamma_E_exact"]
)
history["Gamma_E_numeric"] = np.gradient(
    np.log(history["I2"]), history["s"]
)

aliasing = pd.DataFrame([{
    "R_spec_dealiased": hist256[-1,2],
    "R_spec_without_dealiasing": hist256_noalias[-1,2],
    "absolute_difference":
        abs(hist256[-1,2]-hist256_noalias[-1,2]),
    "I2_damping_off_initial": hist256_nodamp[0,1],
    "I2_damping_off_final": hist256_nodamp[-1,1],
}])
aliasing.to_csv(DATA_DIR / "ch08_caseB_aliasing_energy.csv", index=False)
display(aliasing)

fig, axes = plt.subplots(1, 2, figsize=(7.15,3.05))

axes[0].plot(
    history["s"], history["Gamma_E_exact"],
    color=BLACK, label="spectral identity"
)
axes[0].plot(
    history["s"], history["Gamma_E_numeric"],
    color=DARK, linestyle="--",
    label=r"$d(\ln I_2)/ds$"
)
axes[0].axhline(0.0, color=LIGHT, linewidth=0.8)
axes[0].set_xlabel(r"Dimensionless time, $s$")
axes[0].set_ylabel(r"$\Gamma_E$")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].plot(
    hist256_nodamp[:,0],
    hist256_nodamp[:,1]/hist256_nodamp[0,1]-1.0,
    color=BLACK
)
axes[1].axhline(0.0, color=LIGHT, linewidth=0.8)
axes[1].set_xlabel(r"Dimensionless time, $s$")
axes[1].set_ylabel(r"Relative change in $I_2$ with damping off")
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch08_caseB_identity_checks")
plt.show()

The Case B checks separate three questions:

- convergence of the reported observable;
- sensitivity to de-aliasing;
- consistency with the analytical energy identity.

A visually plausible broadened spectrum is not sufficient if the total-energy behavior violates the model identity.

# Case C — finite-$k$ first-order mode

The canonical dimensional parameters are

$$
\rho=1060\ {\rm kg\,m^{-3}},
\qquad
\mu=3.5\times10^{-3}\ {\rm Pa\,s},
$$

$$
R=4\ {\rm mm},
\qquad
f=1.2\ {\rm Hz},
\qquad
kR=0.2,
$$

with

$$
P=1\ {\rm Pa}.
$$

The exact first-order finite-$k$ mode is evaluated analytically before the second-order mean problem is discretized.

In [ ]:
# Exact Case C first-order mode.
def compliant_first_order(r, R, Omega, k, P):
    r = np.asarray(r, dtype=float)
    lambda_C = np.sqrt(1j*rho*Omega/mu-k**2)
    A = k*P/(Omega*rho)

    Cpsi = (
        -A*iv(0,k*R)
        /(lambda_C*jv(0,lambda_C*R))
    )

    uz = A*iv(0,k*r) + Cpsi*lambda_C*jv(0,lambda_C*r)
    ur = -1j*(
        A*iv(1,k*r)
        + k*Cpsi*jv(1,lambda_C*r)
    )

    uz_r = (
        A*k*iv(1,k*r)
        - Cpsi*lambda_C**2*jv(1,lambda_C*r)
    )

    ur_r = -1j*(
        A*k*0.5*(iv(0,k*r)+iv(2,k*r))
        + k*Cpsi*lambda_C*0.5*(
            jv(0,lambda_C*r)-jv(2,lambda_C*r)
        )
    )

    eta_hat = (
        A*iv(1,k*R)
        + k*Cpsi*jv(1,lambda_C*R)
    )/Omega

    p = P*iv(0,k*r)

    return {
        "lambda_C": lambda_C,
        "Cpsi": Cpsi,
        "uz": uz,
        "ur": ur,
        "uz_r": uz_r,
        "ur_r": ur_r,
        "eta_hat": eta_hat,
        "p": p,
    }

def compatible_impedance(R, Omega, k, P):
    f = compliant_first_order(
        np.array([R]), R, Omega, k, P
    )
    return (
        f["p"][0]-2.0*mu*f["ur_r"][0]
    )/f["eta_hat"]

fC = compliant_first_order(
    np.array([R_C]), R_C, Omega_C, k_C, P_C
)
eta_C = fC["eta_hat"]
Zcompat_C = compatible_impedance(
    R_C, Omega_C, k_C, P_C
)
alpha_C = R_C*np.sqrt(Omega_C/nu)

print("alpha =", alpha_C)
print("eta_hat =", eta_C)
print("Z_compat =", Zcompat_C)

assert abs(alpha_C-6.0445) < 5e-4
assert abs(eta_C-(6.3690-1.7230j)*1e-5)/abs(eta_C) < 2e-4
assert abs(Zcompat_C-(1.4777+0.3984j)*1e4)/abs(Zcompat_C) < 3e-4

# Case C — two radial discretization paths

The Chapter 8 prose states that the second-order mean problem uses a fourth-order finite-difference discretization.

However, during construction of this notebook, the published Case C refinement sequence was found to match the lower-order centered radial solve used as an independent Chapter 6 cross-check, not the genuine fourth-order radial operator implemented below.

The notebook therefore keeps two named paths:

1. **benchmark-reproduction path** — reproduces the tabulated Chapter 8 sequence;
2. **stated fourth-order path** — follows the numerical-method statement in Chapter 8 and Appendix K.

Both converge to essentially the same physical result, but their coarse-grid sequences differ.

This discrepancy is preserved explicitly rather than silently reconciled.

In [ ]:
# Shared Case C second-order forcing.
def caseC_forcing_on_grid(N):
    r = np.linspace(0.0, R_C, N)
    f = compliant_first_order(
        r, R_C, Omega_C, k_C, P_C
    )
    forcing = 0.5*np.real(
        np.conj(f["ur"])*f["uz_r"]
        + np.conj(f["uz"])*(1j*k_C*f["uz"])
    )
    wall = -0.5*np.real(
        np.conj(f["eta_hat"])*f["uz_r"][-1]
    )
    return r, f, forcing, wall

def solve_caseC_benchmark_path(N):
    # Lower-order centered interior discretization that reproduces
    # the published Case C refinement sequence.
    r, f, forcing, wall = caseC_forcing_on_grid(N)
    dr = r[1]-r[0]
    rhs = rho*forcing/mu

    A = lil_matrix((N,N), dtype=float)
    A[0,0] = -4.0/dr**2
    A[0,1] = +4.0/dr**2

    for i in range(1,N-1):
        ri = r[i]
        A[i,i-1] = 1.0/dr**2 - 1.0/(2.0*ri*dr)
        A[i,i]   = -2.0/dr**2
        A[i,i+1] = 1.0/dr**2 + 1.0/(2.0*ri*dr)

    A[-1,-1] = 1.0
    rhs[-1] = wall

    u2 = spsolve(A.tocsr(), rhs)
    Q = 2.0*np.pi*np.trapezoid(r*u2, r)

    # Fourth-order backward wall derivative, matching the Chapter 6 audit.
    du_wall = (
        25*u2[-1]-48*u2[-2]+36*u2[-3]
        -16*u2[-4]+3*u2[-5]
    )/(12*dr)

    return r, f, u2, Q, wall, du_wall

print("Benchmark-reproduction path ready.")

In [ ]:
# General finite-difference weight generator for the stated fourth-order path.
def fd_weights(x0, xs, derivative_order):
    xs = np.asarray(xs, dtype=float)
    n = len(xs)

    A = np.vstack([
        (xs-x0)**p
        for p in range(n)
    ])
    b = np.zeros(n)
    b[derivative_order] = math.factorial(derivative_order)
    return np.linalg.solve(A,b)

def solve_caseC_fourth_order(N):
    # Genuine fourth-order radial discretization.
    r, f, forcing, wall = caseC_forcing_on_grid(N)
    dr = r[1]-r[0]
    rhs = rho*forcing/mu

    A = lil_matrix((N,N), dtype=float)

    # At the centerline, L u = u'' + u'/r -> 2 u''.
    # Use the even extension of the fourth-order five-point second derivative.
    A[0,0] = -60.0/(12.0*dr**2)
    A[0,1] = +64.0/(12.0*dr**2)
    A[0,2] = -4.0/(12.0*dr**2)

    # Fourth-order local five-point approximations for u' and u''.
    for i in range(1,N-1):
        lo = min(max(i-2,0), N-5)
        idx = np.arange(lo,lo+5)

        w1 = fd_weights(r[i], r[idx], 1)
        w2 = fd_weights(r[i], r[idx], 2)
        w = w2 + w1/r[i]

        for j, value in zip(idx,w):
            A[i,j] = value

    A[-1,-1] = 1.0
    rhs[-1] = wall

    u2 = spsolve(A.tocsr(), rhs)
    Q = 2.0*np.pi*np.trapezoid(r*u2, r)

    idx = np.arange(N-5,N)
    w1 = fd_weights(r[-1], r[idx], 1)
    du_wall = np.dot(w1,u2[idx])

    return r, f, u2, Q, wall, du_wall

print("Fourth-order Case C path ready.")

In [ ]:
# Compare both Case C refinement paths with the published table.
published_Q = {
    500: 7.568807e-9,
    1000: 7.568655e-9,
    2000: 7.568617e-9,
    4000: 7.568608e-9,
}
published_tau = {
    500: -2.186351e-4,
    1000: -2.186217e-4,
    2000: -2.186183e-4,
    4000: -2.186175e-4,
}

rows = []
caseC_benchmark_solutions = {}
caseC_fourth_solutions = {}

for N in [500,1000,2000,4000]:
    print("Case C N =", N)

    b = solve_caseC_benchmark_path(N)
    f4 = solve_caseC_fourth_order(N)
    caseC_benchmark_solutions[N] = b
    caseC_fourth_solutions[N] = f4

    # Moving-surface corrections are computed below; here the flux comparison
    # is sufficient to expose the discretization-sequence discrepancy.
    rows.append({
        "N_s": N,
        "published_Q": published_Q[N],
        "benchmark_path_Q": b[3],
        "fourth_order_path_Q": f4[3],
        "benchmark_abs_error":
            abs(b[3]-published_Q[N]),
        "fourth_order_abs_error":
            abs(f4[3]-published_Q[N]),
    })

caseC_compare = pd.DataFrame(rows)
caseC_compare.to_csv(
    DATA_DIR / "ch08_caseC_discretization_comparison.csv",
    index=False
)
display(caseC_compare)

# The benchmark-reproduction path must match the published sequence closely.
assert abs(
    caseC_compare.iloc[-1]["benchmark_path_Q"]
    - published_Q[4000]
)/published_Q[4000] < 2e-6

# Both methods must agree in the converged limit.
assert abs(
    caseC_compare.iloc[-1]["fourth_order_path_Q"]
    - caseC_compare.iloc[-1]["benchmark_path_Q"]
)/caseC_compare.iloc[-1]["benchmark_path_Q"] < 2e-6

In [ ]:
# Visualize the source-level Case C discretization discrepancy.
fig, axes = plt.subplots(1,2,figsize=(7.2,3.1))

axes[0].plot(
    caseC_compare["N_s"],
    1e9*caseC_compare["published_Q"],
    color=BLACK, marker="o", label="published table"
)
axes[0].plot(
    caseC_compare["N_s"],
    1e9*caseC_compare["benchmark_path_Q"],
    color=DARK, linestyle="--", marker="s",
    label="benchmark-reproduction path"
)
axes[0].plot(
    caseC_compare["N_s"],
    1e9*caseC_compare["fourth_order_path_Q"],
    color=MID, linestyle="-.", marker="^",
    label="stated fourth-order path"
)
axes[0].set_xscale("log", base=2)
axes[0].set_xlabel(r"Radial resolution, $N_s$")
axes[0].set_ylabel(r"$10^9 Q_{\mathrm{stream}}$ (m$^3$ s$^{-1}$)")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].loglog(
    caseC_compare["N_s"],
    caseC_compare["benchmark_abs_error"],
    color=DARK, linestyle="--", marker="s",
    label="benchmark path error"
)
axes[1].loglog(
    caseC_compare["N_s"],
    caseC_compare["fourth_order_abs_error"],
    color=MID, linestyle="-.", marker="^",
    label="fourth-order path difference from table"
)
axes[1].set_xlabel(r"Radial resolution, $N_s$")
axes[1].set_ylabel("Absolute difference from published flux")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch08_caseC_discretization_audit")
plt.show()

The left panel shows the issue directly:

- the benchmark-reproduction path tracks the published table;
- the stated fourth-order path reaches the same limiting flux much faster and therefore does not reproduce the published coarse-grid sequence.

This is a **source-level numerical-method inconsistency**, not a disagreement about the converged physical result.

The notebook does not choose one description and silently discard the other.

# Case C moving-surface traction

The complete mean tangential traction contains three $O(\epsilon^2)$ contributions:

$$
\left\langle\sigma_{zr}^{(2)}(R)\right\rangle,
$$

$$
\left\langle
\eta_1\partial_r\sigma_{zr}^{(1)}(R)
\right\rangle,
$$

and

$$
\left\langle
(\partial_z\eta_1)
[
\sigma_{rr}^{(1)}(R)-\sigma_{zz}^{(1)}(R)
]
\right\rangle.
$$

The reference-radius derivative alone is not the complete wall observable.

In [ ]:
# Complete Case C moving-surface traction from the finest benchmark path.
def moving_surface_traction(streaming_tuple):
    r, f, u2, Q, wall, du_wall = streaming_tuple

    lambda_C = f["lambda_C"]
    Cpsi = f["Cpsi"]
    eta_hat = f["eta_hat"]

    Acoef = k_C*P_C/(Omega_C*rho)

    uz_R = f["uz"][-1]
    ur_r_R = f["ur_r"][-1]

    uz_rr_R = (
        Acoef*k_C**2*0.5*(iv(0,k_C*R_C)+iv(2,k_C*R_C))
        - Cpsi*lambda_C**3*0.5*(
            jv(0,lambda_C*R_C)-jv(2,lambda_C*R_C)
        )
    )

    sigma_zr_r_hat = mu*(
        uz_rr_R + 1j*k_C*ur_r_R
    )

    sigma_rr_minus_sigma_zz_hat = (
        2.0*mu*(ur_r_R - 1j*k_C*uz_R)
    )

    # Fluid-on-wall sign by action-reaction.
    gradient = -mu*du_wall

    wall_shift = 0.5*np.real(
        np.conj(eta_hat)*sigma_zr_r_hat
    )

    wall_slope = 0.5*np.real(
        np.conj(1j*k_C*eta_hat)
        * sigma_rr_minus_sigma_zz_hat
    )

    total = gradient + wall_shift + wall_slope

    return {
        "reference_radius_gradient": gradient,
        "wall_shift": wall_shift,
        "wall_slope": wall_slope,
        "total": total,
    }

traction_C = moving_surface_traction(
    caseC_benchmark_solutions[4000]
)
traction_df = pd.DataFrame([traction_C])
traction_df.to_csv(DATA_DIR / "ch08_caseC_traction.csv", index=False)
display(traction_df)

assert abs(
    traction_C["total"] - (-2.186175e-4)
)/2.186175e-4 < 2e-6

In [ ]:
# Traction decomposition figure.
labels = [
    "reference-radius\ngradient",
    "wall shift",
    "wall slope",
    "complete\ntraction",
]
values = [
    traction_C["reference_radius_gradient"],
    traction_C["wall_shift"],
    traction_C["wall_slope"],
    traction_C["total"],
]

fig, ax = plt.subplots(figsize=(6.2,3.3))
ax.bar(
    np.arange(4), values,
    facecolor="white", edgecolor=BLACK, linewidth=0.9
)
ax.axhline(0.0, color=LIGHT, linewidth=0.8)
ax.set_xticks(np.arange(4))
ax.set_xticklabels(labels)
ax.set_ylabel(r"Contribution to $\langle\tau_w^{(2)}\rangle$ (Pa)")
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch08_caseC_traction_decomposition")
plt.show()

# Quadrature is part of the physical definition

Different observables require different measures.

Cross-sectional flux uses

$$
Q
=
2\pi
\int_0^R
r u_z(r)\,dr.
$$

A thin endothelial pillbox uses

$$
A_{\mathrm{EC}}
\int_{R-\delta_{\mathrm{EC}}}^{R}
f_r(r)\,dr.
$$

The cylindrical Jacobian $r$ belongs to the flux definition. It must not be inserted into the near-wall pillbox integral simply because the calculation is performed in cylindrical coordinates after the footprint approximation has already been introduced.

In [ ]:
# Demonstrate that the quadrature measure changes the observable.
r = np.linspace(0.0, 1.0, 2000)
u_demo = 1.0-r**2
f_demo = r*(1.0-r)

Q_correct = 2.0*np.pi*np.trapezoid(r*u_demo, r)
Q_wrong = 2.0*np.pi*np.trapezoid(u_demo, r)

delta = 0.1
mask = r >= 1.0-delta
pillbox_correct = np.trapezoid(np.abs(f_demo[mask]), r[mask])
pillbox_wrong = np.trapezoid(
    r[mask]*np.abs(f_demo[mask]), r[mask]
)

quadrature_audit = pd.DataFrame([{
    "Q_correct_axisymmetric_measure": Q_correct,
    "Q_wrong_without_r": Q_wrong,
    "pillbox_correct_dr_measure": pillbox_correct,
    "pillbox_wrong_with_extra_r": pillbox_wrong,
}])
quadrature_audit.to_csv(
    DATA_DIR / "ch08_quadrature_measure_audit.csv",
    index=False
)
display(quadrature_audit)

This is not a numerical nicety. Changing the measure changes the physical observable being computed.

# Validation hierarchy

A computational result earns interpretation through four complementary tests.

1. **Discretization convergence.**
2. **Analytical or asymptotic recovery.**
3. **Mechanism-off counterfactual.**
4. **Diagnostic identity.**

The three canonical cases can therefore be summarized by their strongest checks:

- **Case A:** tolerance convergence + isotropic Womersley recovery;
- **Case B:** grid/time refinement + de-aliasing sensitivity + energy identity + nonlinearity-off transfer test;
- **Case C:** radial refinement + centerline regularity + moving-wall condition + complete traction + rigid/long-wave recovery.

No one check substitutes for the others.

In [ ]:
# Compact machine-readable verification matrix.
verification_matrix = pd.DataFrame([
    {
        "case": "A",
        "representation": "coupled radial harmonic BVP",
        "convergence": "adaptive BVP tolerance",
        "limit_or_counterfactual": "anisotropy off -> scalar Womersley",
        "identity_or_constraint": "regularity + no slip",
        "status": "checked",
    },
    {
        "case": "B",
        "representation": "periodic axial Fourier evolution",
        "convergence": "N_zeta and Delta_s refinement",
        "limit_or_counterfactual": "quadratic coupling off -> no new modal transfer",
        "identity_or_constraint": "I2 energy identity; de-aliasing audit",
        "status": "checked",
    },
    {
        "case": "C",
        "representation": "finite-k first order + radial mean BVP",
        "convergence": "N_s refinement",
        "limit_or_counterfactual": "rigid/long-wave recovery; second-order sources off",
        "identity_or_constraint": "centerline symmetry + moving-wall boundary",
        "status": "checked with method discrepancy recorded",
    },
])
verification_matrix.to_csv(
    DATA_DIR / "ch08_verification_matrix.csv",
    index=False
)
display(verification_matrix)

# Parameter provenance and sensitivity status

The canonical cases are verification problems.

They are **not** population ranges and are **not** patient-specific calibrations.

| Case | Parameter status | Structural conclusion |
|---|---|---|
| A | dimensionless canonical choice | cross-coupling opens $u_\theta$ and vanishes in isotropic limit |
| B | canonical reduced coefficients | nonlinear transfer can broaden the spectrum while $I_2$ decays |
| C | dimensional canonical tube/fluid values with $P=1$ Pa | moving-wall quadratic closure produces an $O(\epsilon^2)$ mean response |

A physiological range should be introduced only with its measurement definition and source.

# VascuQuest reproducibility demonstration

The purpose of this section is to show a transparent source-to-book mapping and provenance record.

It does **not** replace the canonical cases.

For the deterministic representative subject at the aortic root, the notebook reconstructs:

$$
Q(t)=U(t)A(t),
$$

$$
R=\sqrt{\langle A\rangle_t/\pi},
$$

$$
\alpha
=
R\sqrt{\Omega/\nu},
$$

and the observed wall-motion ratio

$$
\epsilon_{\mathrm{VQ}}
=
\frac{\max_t|R(t)-R|}{R}.
$$

The source geometry is recorded separately.

In [ ]:
# Deterministic VascuQuest mapping example.
SITE = "AorticRoot"

ids_u, U_matrix = load_waveform_matrix(SITE, "U")
ids_a, A_matrix = load_waveform_matrix(SITE, "A")
ids_p, P_matrix = load_waveform_matrix(SITE, "P")
assert np.array_equal(ids_u, ids_a)
assert np.array_equal(ids_u, ids_p)

idx = np.where(ids_u == representative_subject)[0]
if len(idx) != 1:
    raise RuntimeError("Representative subject not found uniquely.")
idx = int(idx[0])

U_values, A_values, p_values = contiguous_prefix(
    U_matrix[idx],
    A_matrix[idx],
    P_matrix[idx],
)

Q_values = U_values*A_values
R_t = np.sqrt(A_values/np.pi)
R_mean = float(np.mean(R_t))
epsilon_vq = float(np.max(np.abs(R_t-R_mean))/R_mean)

hr = float(
    subject_meta.loc[
        subject_meta["subject_id"] == representative_subject,
        "heart_rate_bpm"
    ].iloc[0]
)
Omega_vq = 2.0*np.pi*hr/60.0
alpha_vq = R_mean*np.sqrt(Omega_vq/nu)

mapping_record = {
    "subject_id": representative_subject,
    "site": SITE,
    "samples": int(len(Q_values)),
    "heart_rate_bpm": hr,
    "R_m": R_mean,
    "alpha": alpha_vq,
    "epsilon_VQ": epsilon_vq,
}
pd.DataFrame([mapping_record]).to_csv(
    DATA_DIR / "ch08_vascuquest_mapping_record.csv",
    index=False
)

geo = session.geometry(subject=representative_subject)
geo_rows = [{
    "segment_id": seg.segment_id,
    "length_m": seg.length_m,
    "inlet_radius_m": seg.inlet_radius_m,
    "outlet_radius_m": seg.outlet_radius_m,
} for seg in geo.values]
geo_df = pd.DataFrame(geo_rows)
geo_df.to_csv(
    DATA_DIR / "ch08_representative_geometry.csv",
    index=False
)

display(pd.DataFrame([mapping_record]))

In [ ]:
# Reproducibility-map figure: source waveforms and derived book quantities.
phase = np.arange(len(Q_values), dtype=float)/len(Q_values)

fig, axes = plt.subplots(2,2,figsize=(7.2,5.1), sharex=True)

axes[0,0].plot(phase, p_values, color=BLACK)
axes[0,0].set_ylabel(r"$p$ (mmHg)")
clean_axes(axes[0,0])

axes[0,1].plot(phase, Q_values*1e6, color=BLACK)
axes[0,1].set_ylabel(r"$Q$ (mL s$^{-1}$)")
clean_axes(axes[0,1])

axes[1,0].plot(phase, 1e3*R_t, color=BLACK)
axes[1,0].axhline(1e3*R_mean, color=DARK, linestyle="--")
axes[1,0].set_xlabel(r"Normalized time, $t/T$")
axes[1,0].set_ylabel(r"$R(t)$ (mm)")
clean_axes(axes[1,0])

axes[1,1].axis("off")
summary = (
    rf"$R={1e3*R_mean:.3f}$ mm" "\n"
    rf"$\alpha={alpha_vq:.3f}$" "\n"
    rf"$\epsilon_{{\mathrm{{VQ}}}}={epsilon_vq:.4f}$"
)
axes[1,1].text(
    0.05,0.72,summary,
    transform=axes[1,1].transAxes,
    va="top", ha="left", fontsize=10
)

fig.tight_layout()
save_figure(fig, "ch08_vascuquest_reproducibility_map")
plt.show()

The VascuQuest figure demonstrates data provenance and notation translation, not model validation.

The reader-facing notebook uses $p$, $Q$, $R$, $\alpha$, and $\epsilon_{\mathrm{VQ}}$. The database-native signal labels remain inside the ingestion layer.

# Optional higher-order CGL branch

Chapter 8 stops the verified compliant workflow at second order.

A quantitative CGL calculation would require:

- a direct neutral mode;
- an adjoint mode;
- second-order mean correction;
- second harmonic;
- Fredholm solvability projection;
- normalization;
- step-halving of dispersion curvature;
- denominator-conditioning checks.

Because that complete calculation is not part of the current book benchmark, this notebook does **not** invent numerical values for

$$
\sigma,\qquad
\xi,\qquad
\beta,
$$

or for the Benjamin--Feir--Newell diagnostic.

# What the reader should learn

1. **Representation is chosen before discretization.** The coordinate, field, and recovery limit determine the numerical method.

2. **Transform convention is part of the model implementation.** A normalization error propagates directly into reconstructed nonlinear observables.

3. **Analytical differentiation is a valuable numerical reference when available.**

4. **Convergence is necessary but not sufficient.** A solver can converge to the wrong boundary-value problem.

5. **Mechanism-off tests are physical verification tests.** They check whether the claimed mechanism actually controls the reported effect.

6. **Diagnostic identities constrain interpretation.** Case B must respect its energy law; radial problems must respect regularity and wall conditions.

7. **Quadrature measure is part of the observable definition.** Flux, near-wall volume integration, and traction do not use interchangeable measures.

8. **Parameter provenance must remain explicit.** Cases A–C are verification settings, not physiological population ranges.

9. **VascuQuest can make provenance and mapping reproducible without replacing canonical verification.**

10. **The current Chapter 8 source contains a numerical-method inconsistency in Case C.** The stated fourth-order method and the published coarse-grid refinement sequence are not the same discretization, although both converge to the same limiting result.

# Chapter-enrichment candidates

The notebook produces eight principal figures.

**Candidate 1 — Fourier reconstruction contract.**  
Notebook-first. Useful for readers who need an executable demonstration of the transform normalization and $2M$ nonlinear bandwidth.

**Candidate 2 — analytical versus numerical Womersley derivative.**  
Notebook-first verification material.

**Candidate 3 — Case A verification and isotropic recovery.**  
Strong notebook result; Chapter 8 already reports the benchmark numerically, so a book figure is optional.

**Candidate 4 — nonlinear reconstruction spectrum.**  
Notebook-first because Chapters 4 and 7 already own the physical interpretation.

**Candidate 5 — Case B energy-identity checks.**  
Potential book candidate if Chapter 8 needs a visual distinction between convergence, de-aliasing, and the structural energy constraint.

**Candidate 6 — Case C discretization audit.**  
Strong editorial/scientific candidate. It exposes a source inconsistency that should be resolved before final publication: stated fourth-order method versus published coarse-grid sequence.

**Candidate 7 — Case C moving-surface traction decomposition.**  
Potential replacement or cross-reference figure; Chapter 6 owns the physical mechanics.

**Candidate 8 — VascuQuest reproducibility map.**  
Strong notebook figure and possible repository/documentation figure. It demonstrates the source-to-book mapping without pretending to validate the canonical models.

No figure is promoted automatically.

In [ ]:
# Final reproducibility manifest.
manifest = {
    "book": "Nonlinear Arterial Hemodynamics",
    "chapter": 8,
    "chapter_title": "Computing the Theory",
    "vascuquest_git_ref": VQ_GIT_REF,
    "pwdb_record_id": PWDB_RECORD_ID,
    "pwdb_doi": PWDB_DOI,
    "rho_kg_m3": rho,
    "mu_Pa_s": mu,
    "nu_m2_s": nu,
    "caseA": {
        "alpha": alpha_A,
        "m": m_A,
        "a_m": a_A,
        "A_ztheta": A_ztheta,
        "A_thetaz": A_thetaz,
        "A_thetatheta": A_thetatheta,
        "finest_max_abs_Uz":
            float(caseA_df.iloc[-1]["max_abs_Uz"]),
        "finest_max_abs_Utheta":
            float(caseA_df.iloc[-1]["max_abs_Utheta"]),
    },
    "caseB": {
        "L_g": L_g,
        "alpha": alpha_B,
        "b_avg": b_B,
        "g_avg": g_B,
        "finest_I2_10":
            float(caseB_df.iloc[-1]["I2_10"]),
        "finest_R_spec_10":
            float(caseB_df.iloc[-1]["R_spec_10"]),
    },
    "caseC": {
        "R_m": R_C,
        "f_Hz": f_C,
        "kR": kR_C,
        "P_Pa": P_C,
        "benchmark_path_Q_4000":
            float(caseC_compare.iloc[-1]["benchmark_path_Q"]),
        "fourth_order_path_Q_4000":
            float(caseC_compare.iloc[-1]["fourth_order_path_Q"]),
        "complete_tau_w2_Pa":
            float(traction_C["total"]),
        "method_discrepancy_recorded": True,
    },
    "representative_subject_id": representative_subject,
    "representative_site": SITE,
    "representative_mapping": mapping_record,
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "execution_status": "completed to this cell",
    "qualification": (
        "Cases A-C are canonical verification problems. "
        "VascuQuest is used for deterministic source-to-book mapping and provenance, "
        "not as external experimental validation. "
        "The Case C stated-method/tabulated-sequence inconsistency is preserved explicitly."
    ),
}

(META_DIR / "reproducibility_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print(json.dumps(manifest, indent=2))
print("\nGenerated PDF figures:")
for path in sorted(FIG_DIR.glob("*.pdf")):
    print(" -", path.name)

# Reproducibility record

A successful **Run all** execution writes:

- B&W vector PDF figures and high-resolution PNG previews;
- Fourier reconstruction and nonlinear-bandwidth diagnostics;
- Womersley derivative verification;
- Case A tolerance convergence and isotropic recovery;
- Case B grid/time refinement, de-aliasing, and energy-identity checks;
- Case C first-order verification;
- both the benchmark-reproduction and stated fourth-order streaming paths;
- the complete moving-surface traction decomposition;
- the verification matrix;
- the deterministic VascuQuest source-to-book mapping;
- representative source geometry;
- VascuQuest/PWDB verification metadata;
- the final reproducibility manifest.

The notebook contains no hidden physiological calibration, no unverified CGL coefficients, and no silent reconciliation of the Case C numerical-method discrepancy.